# Course 1: From Beginner to an AI Research Lab
## Session 6: Transformers and modern models

**Goal:** write attention and check it, then run a controlled ablation over three seeds on a small GPT.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. Attention, with two checks

`Attention(Q, K, V) = softmax(Q Kᵀ / sqrt(d)) V`

In [ ]:
import math
import torch

def attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(k.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    return torch.softmax(scores, dim=-1) @ v

torch.manual_seed(0)
T_ = 5
x = torch.randn(1, T_, 16)
mask = torch.tril(torch.ones(T_, T_))
out = attention(x, x, x, mask)

# check 1: matches PyTorch's own implementation
ref = torch.nn.functional.scaled_dot_product_attention(x, x, x, attn_mask=mask.bool())
print(torch.allclose(out, ref, atol=1e-6))

# check 2: changing the last token leaves earlier outputs alone
x2 = x.clone()
x2[0, -1] += 1.0
out2 = attention(x2, x2, x2, mask)
print(torch.allclose(out[0, :-1], out2[0, :-1]))
assert torch.allclose(out, ref, atol=1e-6) and torch.allclose(out[0, :-1], out2[0, :-1])

### 2. A small GPT you can train on a CPU
nanoGPT needs a GPU for its lab runs (last cell). To practise the experimental method right now, this notebook uses a GPT with about 100,000 parameters and a toy Bangla language built from a small grammar. The best possible loss on that language is known, which makes results easy to judge.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

In [ ]:
floor = math.log(len(ALL_SENTENCES)) / 6
cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=32)
base = train_lm(cfg, steps=300)
print("example:", " ".join(train_sents[0])); print(f"baseline validation loss {val_loss(base):.3f} | best possible {floor:.3f}")

### 3. One ablation: remove the position embedding
A controlled experiment changes one thing. Here the position embedding is switched off and everything else stays the same: data, steps, learning rate, model size.

Prediction before you run it: attention with a causal mask can still see which tokens came earlier, but not in what order. Will the model still learn where a sentence ends?

In [ ]:
class NoPosGPT(GPT):
    def forward(self, idx, targets=None, return_hidden=False):
        self.pos.weight.data.zero_()                         # the ablation: position information removed
        return super().forward(idx, targets, return_hidden)

def train_variant(cls, seed, steps=300):
    torch.manual_seed(seed); model = cls(cfg); opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for _ in range(steps):                                   # same number of steps for every run
        xb, yb = get_batch(train_ids, cfg.block_size, 32)
        _, loss = model(xb, yb); opt.zero_grad(); loss.backward(); opt.step()
        if cls is NoPosGPT: model.pos.weight.data.zero_()
    return val_loss(model)

### 4. Three seeds each, then a table with intervals

In [ ]:
import numpy as np
from scipy import stats
results = {"baseline": [train_variant(GPT, s) for s in (1, 2, 3)], "no position embedding": [train_variant(NoPosGPT, s) for s in (1, 2, 3)]}
def summarize(v):
    v = np.array(v); half = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v)); return v.mean(), half
print(f"{'variant':24s} runs                      mean ± 95% interval")
for name, v in results.items():
    m, h = summarize(v); print(f"{name:24s} {np.round(v, 3)}   {m:.3f} ± {h:.3f}")
diff = np.mean(results["no position embedding"]) - np.mean(results["baseline"])
print(f"difference in means: {diff:+.3f} (best possible loss is {floor:.3f})")

Write down what the table lets you say, in one sentence, and what it does not. With three seeds the intervals are wide. If the two intervals overlap, the honest statement is 'no clear difference at this scale'. If they are far apart, say how far and at what number of steps. Session 9 covers this properly.

### 5. What makes this a fair comparison (checklist)
- Same data and same split.
- Same number of steps, and results compared at equal steps.
- Same seeds for both variants.
- One change only.
- Every run saved to its own folder, with its config.

### 6. The lab on nanoGPT (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU notebook (Kaggle or Colab) and internet to clone the repository.
#   git clone https://github.com/karpathy/nanoGPT && cd nanoGPT
#   python data/shakespeare_char/prepare.py          # for Bangla: replace data/shakespeare_char/input.txt with your own text first
#   python train.py config/train_shakespeare_char.py
#   python sample.py --out_dir=out-shakespeare-char
#
# The ablation: copy model.py, and either remove the position embedding or replace attention weights with a uniform average over earlier tokens.
# Seeds: nanoGPT fixes its seed inside train.py (find the torch.manual_seed line). Add a `seed` variable beside the other settings
# at the top of the file, use it on that line, then pass --seed=1, --seed=2, --seed=3 with a new --out_dir for every run.
#
# The README reports about 3 minutes on an A100 for this model, with a best validation loss near 1.47. A T4 is several times slower:
# lower --max_iters until six runs fit your quota, and compare runs only at equal steps.

### TODO
1. Run a second ablation here: replace learned attention with a uniform average over earlier tokens. Predict the result first.
2. Do the nanoGPT lab: baseline and one ablation, three seeds each, one table.
3. Read `model.py` and mark where the attention, the feed-forward network, the residual connections and the position embeddings are.